In [ ]:
from cropgen.external_interfaces.LabelStudioInterface import LabelStudioInterface
from cropgen.processing.helpers.PairingErrors import PairingError
from cropgen.external_interfaces.OracleBucketInterface import OracleBucketInterface
from PIL import Image
from tqdm.auto import tqdm
from cropgen.shared.PathBundle import PathBundle
from cropgen.processing.AnnotatedPage import AnnotatedPage

paths = PathBundle()

OracleBucketInterface.from_env(paths).update()

LabelStudioInterface.update_conditional(paths)

lsi = LabelStudioInterface(paths)

simplified_tasks = lsi.simplified_tasks


for task in tqdm(simplified_tasks, desc = "pairing check"):
    task_id = task["id"]
    try:
        img_path = paths.get_image_path_from_task(task)
        if img_path is None:
            print(
                f"(Task {task_id}) > No se ha podido encontrar el path a la imagen de esta tarea."
            )
            continue
        img = Image.open(img_path)
        for ann in task["annotations"]:
            try:
                Ann = AnnotatedPage(ann, img, unrotate=False, usernames_labelstudio= lsi.usernames)
                if not (len(Ann.image_boxes) > 0):
                    print(f"La tarea {task_id} no tiene anotaciones.")
            except PairingError as E:
                print(E)
                print(type(E).__name__)
            except Exception as E:
                print(
                    f"Task {task_id} - Error durante la creación de la AnnotatedPage: {E}"
                )
                raise E
    except Exception as E:
        print(
            f"Task {task_id} - Error durante la creación de los parámetros de AnnotatedPag: {E}"
        )
        raise E

print(f"Comprobadas las {len(simplified_tasks)} tareas.")


